# 176. Second Highest Salary

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** database, subquery, NULL
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/second-highest-salary/)

```
Table: Employee
+-------------+------+
| Column Name | Type |
+-------------+------+
| id          | int  |
| salary      | int  |
+-------------+------+
id is the primary key.
```

Write a solution to find the **second highest distinct** salary from the `Employee`
table. If there is no second highest salary, return `null`.

The result column must be called `SecondHighestSalary`.

---

### Example 1

```
Employee:                        Output:
+----+--------+                  +---------------------+
| id | salary |                  | SecondHighestSalary |
+----+--------+                  +---------------------+
| 1  | 100    |                  | 200                 |
| 2  | 200    |                  +---------------------+
| 3  | 300    |
+----+--------+
```

### Example 2

```
Employee:                        Output:
+----+--------+                  +---------------------+
| id | salary |                  | SecondHighestSalary |
+----+--------+                  +---------------------+
| 1  | 100    |                  | null                |
+----+--------+                  +---------------------+
```

---

Read example 2 again, because it is the whole problem. It does not say "return no
rows". It says return **one row containing `null`** - and the obvious query returns
zero rows instead, which is a different thing and marked wrong.

## Before you write anything

**1.** Write the obvious version first:

```sql
SELECT DISTINCT salary AS SecondHighestSalary
FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1
```

Run it on example 1 - correct. Now run it on example 2 with `show`. Count the rows you
get back, and count the rows the expected output has. **They are not the same number.**
Write down the difference in your own words; it is the entire problem.

**2.** "Zero rows" and "one row containing `null`" are different results. A report that
prints one blank cell and a report that prints nothing at all look different to whoever
reads it, and only one of them says "we checked, and there isn't one". Which one does
the statement ask for?

**3.** There is a neat trick for turning "possibly no rows" into "always exactly one
row, possibly null": wrap the whole thing in an outer `SELECT` as a **scalar subquery**.

```sql
SELECT ( <your query here> ) AS SecondHighestSalary
```

Why does that produce a row when the inner query is empty? (You met the rule in #175
route B: a scalar subquery with no rows *is* `NULL`.) Say what the outer `SELECT` has in
its `FROM` clause, and why that is what guarantees exactly one row.

**4.** There is a second solution that gets it right with no trick at all:
`MAX(salary)` of the salaries that are strictly below the overall maximum. Why does
`MAX` of an empty set return `NULL` rather than no rows? (Because an aggregate with no
`GROUP BY` always produces exactly one row - which is the same fact as question 3, from
a different angle.)

**5.** **Distinct.** If three people all earn 300 and one earns 100, what is the second
highest *distinct* salary? Where does `DISTINCT` have to go for `LIMIT 1 OFFSET 1` to
be right - and does the `MAX`-based version need it at all? Answer for both.

**6.** What should the answer be when the table is **empty**? Work out what each of your
two versions returns, and check they agree.

## Two routes

**A - `MAX` of everything below the maximum** *(write this first)*

```sql
SELECT MAX(salary) AS SecondHighestSalary
FROM Employee
WHERE salary < (SELECT MAX(salary) FROM Employee)
```

No `DISTINCT` and no `LIMIT`. When every salary is equal, the `WHERE` matches nothing,
`MAX` over nothing is `NULL`, and an aggregate with no `GROUP BY` still returns exactly
one row - so you get the required `null` for free, by construction rather than by
patching.

Duplicates are handled automatically: `MAX` does not care how many rows hold the value.

**B - `LIMIT`/`OFFSET`, wrapped**

```sql
SELECT (
    SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1
) AS SecondHighestSalary
```

The inner query is the natural way to say "skip one, take one", and it needs `DISTINCT`
so that duplicate top salaries do not eat the offset. The outer `SELECT` with no `FROM`
is what converts "no rows" into "one null row" - and generalises: `OFFSET 2` gives you
the third highest, which is **#177 Nth Highest Salary**.

> **"No rows" is not "null".** They are different answers to different questions, and
> SQL will hand you the first one when you meant the second unless you ask carefully.
> Aggregates without `GROUP BY` and scalar subqueries both always produce exactly one
> row - remember that pair and you have the tool for every "return null if there isn't
> one" problem there is.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Employee (id INTEGER, salary INTEGER);"""

EXPECTED_COLUMNS = ['SecondHighestSalary']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example 1", '''
INSERT INTO Employee VALUES (1,100), (2,200), (3,300);
''', [(200,)])

check("*** the LeetCode example 2: ONE row containing null ***", '''
INSERT INTO Employee VALUES (1,100);
''', [(None,)])

check("*** an empty table is still one null row ***", '', [(None,)])

check("question 5: every salary is identical", '''
INSERT INTO Employee VALUES (1,300), (2,300), (3,300);
''', [(None,)])

check("question 5: the top salary is duplicated", '''
INSERT INTO Employee VALUES (1,300), (2,300), (3,200), (4,100);
''', [(200,)])

check("the second salary is duplicated too", '''
INSERT INTO Employee VALUES (1,300), (2,200), (3,200), (4,100);
''', [(200,)])

check("exactly two distinct salaries", '''
INSERT INTO Employee VALUES (1,50), (2,90);
''', [(50,)])

check("rows inserted in ascending order", '''
INSERT INTO Employee VALUES (1,100), (2,200), (3,300), (4,400);
''', [(300,)])

check("rows inserted in descending order", '''
INSERT INTO Employee VALUES (1,400), (2,300), (3,200), (4,100);
''', [(300,)])

check("negative and zero salaries", '''
INSERT INTO Employee VALUES (1,0), (2,-100), (3,-50);
''', [(-50,)])

check("one row only, salary zero", '''
INSERT INTO Employee VALUES (1,0);
''', [(None,)])

## After it passes

- **Watch the unwrapped version fail.** Run the bare `LIMIT 1 OFFSET 1` query with
  `show` on the single-employee dataset. Zero rows. Then wrap it and run it again. One
  null row. That is the whole problem, in two commands.
- **Generalise it to #177.** `OFFSET 1` is the second highest; `OFFSET N-1` is the Nth.
  Write **#177 Nth Highest Salary** as a function taking `N` - it is your route B with
  one substitution, and the `null` behaviour you just built carries straight over.
- **Then use a window function.** `DENSE_RANK() OVER (ORDER BY salary DESC)` gives every
  distinct salary a rank; filter for rank 2. Note what happens on example 2: zero rows
  again, so you still need the wrapper. Understanding *why the wrapper is still needed*
  is the point - the null problem is about the shape of the result, not about which tool
  you used.
- **Say what `DENSE_RANK` does that `RANK` does not**, using the "top salary is
  duplicated" dataset. That distinction is the whole of **#185**, so get it straight
  here where it is cheap.
- Siblings: **#177 Nth Highest Salary** (this, parameterised), #178 Rank Scores
  (`DENSE_RANK` as the answer rather than as an aside), **#185 Department Top Three
  Salaries** (this, per group, at N=3), #184 Department Highest Salary.